# Role inference for call transcripts

Этот ноутбук решает первый этап pipeline: **определение ролей в телефонном звонке**.

## Цель
Для каждого звонка определить:
- кто менеджер;
- кто клиент;
- был ли секретарь / gatekeeper;
- удалось ли менеджеру выйти на ЛПР;
- какой общий тип у звонка.

## Почему здесь используется LLM
Для этой задачи лучше подходит instruction-LLM, потому что роли определяются по **содержанию диалога**, а не только по speaker label.

Рекомендуемая стартовая модель:
- `Qwen/Qwen2.5-3B-Instruct`

Если нужно более стабильное качество:
- `Qwen/Qwen2.5-7B-Instruct`


In [1]:
# Если библиотеки ещё не установлены, раскомментируйте строку ниже:
# !pip install -q transformers accelerate sentencepiece torch pandas tqdm

In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm

## 1. Конфигурация

Здесь задаются:
- путь к данным;
- имя модели;
- параметры генерации.


In [3]:
# Путь к JSONL с транскриптами
DATA_PATH = Path("data/aligned_speakers_whisper_large_v3_plus_diarization.jsonl")

# Рекомендуемая стартовая модель
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Параметры генерации
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0

# Сколько блоков диалога отправляем в prompt
MAX_BLOCKS_FOR_PROMPT = 40

## 2. Схемы данных

Используем dataclass, чтобы код был прозрачнее и удобнее для отладки.


In [4]:
@dataclass
class SpeakerBlock:
    speaker: str
    start: float
    end: float
    text: str


@dataclass
class CallRecord:
    call_id: str
    path: str
    duration_sec: float
    speaker_blocks: List[SpeakerBlock]


@dataclass
class RoleInferenceResult:
    call_id: str
    manager_speaker: Optional[str]
    client_speakers: List[str]
    gatekeeper_present: Optional[bool]
    decision_maker_present: Optional[bool]
    call_topology: str
    confidence: float
    evidence: List[Dict[str, Any]]
    raw_model_output: str

## 3. Загрузка данных

Ожидается формат JSONL:
- одна строка = один звонок;
- внутри есть `speaker_blocks`;
- каждый блок содержит `speaker`, `start`, `end`, `text`.


In [5]:
def load_calls_from_jsonl(path: Path) -> List[CallRecord]:
    """Загружает звонки из JSONL и переводит их в удобные Python-структуры."""
    calls: List[CallRecord] = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue

            row = json.loads(line)

            blocks: List[SpeakerBlock] = []
            for block in row.get("speaker_blocks", []):
                speaker = block.get("speaker")
                text = (block.get("text") or "").strip()

                if speaker is None or not text:
                    continue

                blocks.append(
                    SpeakerBlock(
                        speaker=speaker,
                        start=float(block.get("start", 0.0)),
                        end=float(block.get("end", 0.0)),
                        text=text,
                    )
                )

            calls.append(
                CallRecord(
                    call_id=row["id"],
                    path=row.get("path", ""),
                    duration_sec=float(row.get("duration_sec", 0.0)),
                    speaker_blocks=blocks,
                )
            )

    return calls


calls = load_calls_from_jsonl(DATA_PATH)
print(f"Loaded calls: {len(calls)}")

Loaded calls: 103


## 4. Нормализация текста

На этом шаге:
- убираем лишние пробелы;
- склеиваем переносы строк;
- получаем более чистый текст для правил и LLM.


In [6]:
def normalize_text(text: str) -> str:
    """Убирает лишние пробелы и переводит текст в более чистый вид."""
    text = text.replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_call(call: CallRecord) -> CallRecord:
    """Применяет нормализацию ко всем репликам звонка."""
    normalized_blocks: List[SpeakerBlock] = []

    for block in call.speaker_blocks:
        normalized_blocks.append(
            SpeakerBlock(
                speaker=block.speaker,
                start=block.start,
                end=block.end,
                text=normalize_text(block.text),
            )
        )

    return CallRecord(
        call_id=call.call_id,
        path=call.path,
        duration_sec=call.duration_sec,
        speaker_blocks=normalized_blocks,
    )


calls = [normalize_call(call) for call in calls]

## 5. Precheck перед LLM

Здесь идут простые проверки, чтобы:
- отсеять пустые или сломанные звонки;
- определить автоответчик;
- собрать примитивные эвристики по ролям.


In [7]:
AUTO_RESPONDER_PATTERNS = [
    r"абонент .* не может ответить",
    r"оставьте сообщение",
    r"телефон занят",
    r"попробуйте перезвонить",
    r"после звукового сигнала",
]

MANAGER_HINTS = [
    "здравствуйте",
    "добрый день",
    "это",
    "компания",
    "подскажите",
    "могу переговорить",
    "кто у вас занимается",
    "звоню",
]


def is_broken_call(call: CallRecord) -> bool:
    """Проверяет, что звонок слишком короткий или почти пустой."""
    if len(call.speaker_blocks) == 0:
        return True

    full_text = " ".join(block.text for block in call.speaker_blocks).strip()
    return len(full_text) < 10


def is_autoresponder(call: CallRecord) -> bool:
    """Проверяет, похож ли звонок на автоответчик."""
    full_text = " ".join(block.text.lower() for block in call.speaker_blocks)
    return any(re.search(pattern, full_text) for pattern in AUTO_RESPONDER_PATTERNS)


def get_unique_speakers(call: CallRecord) -> List[str]:
    """Возвращает список уникальных speaker labels в звонке."""
    return sorted({block.speaker for block in call.speaker_blocks if block.speaker})


def calculate_manager_hint_scores(call: CallRecord) -> Dict[str, int]:
    """Грубая эвристика: у какого спикера больше типичных фраз менеджера."""
    scores: Dict[str, int] = {}

    for block in call.speaker_blocks:
        speaker = block.speaker
        text = block.text.lower()

        if speaker not in scores:
            scores[speaker] = 0

        for hint in MANAGER_HINTS:
            if hint in text:
                scores[speaker] += 1

    return scores

## 6. Подготовка диалога для модели

Мы явно нумеруем реплики, чтобы модель могла ссылаться на них в `evidence`.


In [8]:
def format_dialogue_for_prompt(
    call: CallRecord, max_blocks: int = MAX_BLOCKS_FOR_PROMPT
) -> str:
    """Готовит красивый текст диалога для prompt."""
    lines: List[str] = []

    for turn_id, block in enumerate(call.speaker_blocks[:max_blocks]):
        line = f"[{turn_id}] {block.speaker} ({block.start:.2f}-{block.end:.2f}): {block.text}"
        lines.append(line)

    return "\n".join(lines)

## 7. JSON-схема ответа модели

Важно заставить модель возвращать **строгий JSON**, чтобы результат было легко парсить.


In [9]:
ROLE_SCHEMA_DESCRIPTION = """
Верни только JSON без markdown и без пояснений.

Формат:
{
  "manager_speaker": "SPEAKER_00" | "SPEAKER_01" | null,
  "client_speakers": ["SPEAKER_00", "SPEAKER_01", ...],
  "gatekeeper_present": true | false | null,
  "decision_maker_present": true | false | null,
  "call_topology": "manager_to_client" | "manager_to_gatekeeper" | "manager_to_gatekeeper_then_decision_maker" | "voicemail" | "broken_call" | "unclear",
  "confidence": 0.0,
  "evidence": [
    {"speaker": "SPEAKER_00", "turn_id": 0, "reason": "..."}
  ]
}
"""

## 8. Prompt для role inference

Здесь описана сама задача для LLM.


In [10]:
# def build_role_inference_prompt(call: CallRecord) -> str:
#     """Создаёт prompt для определения ролей в звонке."""
#     dialogue = format_dialogue_for_prompt(call)

#     prompt = f"""
# Ты анализируешь транскрипт телефонного звонка на русском языке.

# Твоя задача:
# определить, кто в звонке выполняет роль менеджера,
# кто относится к стороне клиента,
# есть ли секретарь / gatekeeper,
# и удалось ли менеджеру выйти на ЛПР.

# Как интерпретировать роли:
# - Менеджер — инициирует продажу, представляется, называет компанию, просит соединить,
#   задаёт квалификационные вопросы, презентует продукт, предлагает следующий шаг.
# - Gatekeeper / секретарь — фильтрует доступ к нужному человеку,
#   спрашивает "кто вы", "по какому вопросу", отвечает "его нет", "соединяю", "отправьте на почту".
# - ЛПР / decision maker — человек, который может обсуждать продукт по существу
#   и принимать решение или участвовать в согласовании.

# Правила:
# 1. Не делай вывод по speaker id. Опирайся на содержание реплик.
# 2. Если звонок похож на автоответчик или broken call, укажи это в call_topology.
# 3. Если информации недостаточно, лучше вернуть null или unclear.
# 4. Не придумывай лишнего.
# 5. Верни только JSON.

# {ROLE_SCHEMA_DESCRIPTION}

# Транскрипт:
# {dialogue}
# """.strip()

#     return prompt

In [11]:
def build_role_inference_prompt(call: CallRecord) -> str:
    dialogue = format_dialogue_for_prompt(call)

    prompt = f"""
Ты анализируешь транскрипт телефонного звонка на русском языке.

Задача:
по содержанию разговора определить:
1) кто является менеджером,
2) кто относится к стороне клиента,
3) был ли в звонке gatekeeper / секретарь,
4) присутствовал ли decision maker / ЛПР,
5) определить тип звонка (call_topology).

Важно:
- Не определяй роли по speaker label. Нельзя считать, что SPEAKER_00 всегда менеджер.
- Опирайся только на содержание реплик.
- Если данных недостаточно, лучше вернуть null или unclear, чем выдумывать.
- Верни только JSON, без markdown и без пояснений.

====================
ОПРЕДЕЛЕНИЯ РОЛЕЙ
====================

Менеджер:
Это участник звонка, который выполняет большую часть следующих действий:
- инициирует разговор;
- представляется;
- называет компанию;
- объясняет цель звонка;
- просит соединить с нужным человеком;
- задаёт квалификационные вопросы;
- презентует продукт или услугу;
- предлагает следующий шаг: тест, встречу, отправку информации, повторный звонок.

Gatekeeper / секретарь:
Это участник на стороне клиента, который:
- ограничивает доступ к другому человеку;
- не обсуждает продукт по существу;
- отвечает фразами типа:
  "его нет",
  "она занята",
  "соединяю",
  "кто вы",
  "по какому вопросу",
  "пришлите на почту",
  "я передам",
  "перезвоните позже".

Decision maker / ЛПР:
Это участник на стороне клиента, который:
- обсуждает предложение по существу;
- отвечает на вопросы о процессе, рекламе, продукте, стоимости, тесте, результате;
- может принимать решение, согласовывать следующий шаг или явно влияет на решение;
- не просто фильтрует звонок и не только перенаправляет.

====================
КАК ОПРЕДЕЛЯТЬ ПОЛЯ
====================

1. manager_speaker
Выбери speaker, который наиболее явно является менеджером.
Если такого speaker нельзя определить достаточно уверенно, верни null.

2. client_speakers
Это все спикеры, кроме менеджера, которые относятся к стороне клиента.
Если менеджер не определён, client_speakers можно вернуть как пустой список или список вероятных клиентских спикеров только если это очевидно.

3. gatekeeper_present
Верни true, если в разговоре был участник, который выполнял функцию gatekeeper / секретаря.
Верни false, если такого участника не было.
Верни null, если по транскрипту это невозможно понять.

4. decision_maker_present
Верни true, если менеджер разговаривал с человеком, который обсуждал предложение по существу или мог принимать / согласовывать решение.
Верни false, если такого человека в звонке не было.
Верни null, если это невозможно понять по транскрипту.

====================
КАК ОПРЕДЕЛЯТЬ CALL_TOPOLOGY
====================

Используй только одно из следующих значений:
- "manager_to_client"
- "manager_to_gatekeeper"
- "manager_to_gatekeeper_then_decision_maker"
- "voicemail"
- "broken_call"
- "unclear"

Никогда не придумывай другие значения.

Правила:
1. "voicemail"
Выбери это значение, если звонок похож на автоответчик, голосовое меню, запись, шаблонное авто-сообщение, либо реплики не похожи на живой двусторонний разговор.

2. "broken_call"
Выбери это значение, если звонок слишком короткий, оборванный, пустой, шумный или по нему нельзя восстановить структуру разговора.

3. "manager_to_gatekeeper_then_decision_maker"
Выбери это значение, если сначала менеджер общается с gatekeeper / секретарём, а затем в том же звонке появляется decision maker / ЛПР.

4. "manager_to_gatekeeper"
Выбери это значение, если менеджер общается только с gatekeeper / секретарём и до decision maker не доходит.

5. "manager_to_client"
Выбери это значение, если менеджер сразу общается с клиентом / ЛПР без gatekeeper.

6. "unclear"
Выбери это значение только если информации действительно недостаточно и ни один из вариантов выше нельзя выбрать уверенно.

Важно:
- Если есть gatekeeper_present = true и decision_maker_present = false, то обычно call_topology = "manager_to_gatekeeper".
- Если есть gatekeeper_present = true и decision_maker_present = true, то обычно call_topology = "manager_to_gatekeeper_then_decision_maker".
- Если gatekeeper_present = false и decision_maker_present = true, то обычно call_topology = "manager_to_client".
- Если topology не может быть надёжно выведена, верни "unclear".

====================
CONFIDENCE
====================

Верни confidence от 0.0 до 1.0.

Используй высокий confidence (0.8-1.0), только если:
- роли хорошо видны из текста,
- есть явные реплики-основания,
- topology определяется без противоречий.

Используй средний confidence (0.5-0.79), если:
- вывод правдоподобен, но есть неоднозначность.

Используй низкий confidence (0.0-0.49), если:
- звонок короткий,
- шумный,
- неполный,
- есть противоречия,
- выбран unclear / broken_call.

Если manager_speaker = null или call_topology = "unclear", confidence обычно не должен быть высоким.

====================
EVIDENCE
====================

В evidence верни 1-3 наиболее важных реплики, на которых основан вывод.
Каждый элемент evidence должен иметь поля:
- speaker
- turn_id
- reason

reason должен быть коротким и конкретным, например:
- "представляется и инициирует разговор"
- "фильтрует доступ к другому человеку"
- "обсуждает предложение по существу"
- "похоже на автоответчик"

====================
ФОРМАТ ОТВЕТА
====================

Верни только JSON строго такого вида:

{{
  "manager_speaker": "SPEAKER_00" | "SPEAKER_01" | null,
  "client_speakers": ["SPEAKER_00", "SPEAKER_01"],
  "gatekeeper_present": true | false | null,
  "decision_maker_present": true | false | null,
  "call_topology": "manager_to_client" | "manager_to_gatekeeper" | "manager_to_gatekeeper_then_decision_maker" | "voicemail" | "broken_call" | "unclear",
  "confidence": 0.0,
  "evidence": [
    {{
      "speaker": "SPEAKER_00",
      "turn_id": 0,
      "reason": "..."
    }}
  ]
}}

Запрещено:
- добавлять любые другие поля;
- возвращать текст вне JSON;
- придумывать новые значения call_topology;
- ставить в decision_maker_present строку, имя человека или объект;
- объяснять ответ после JSON.

====================
ТРАНСКРИПТ
====================

{dialogue}
""".strip()

    return prompt

## 9. Загрузка модели

Ниже — базовый вариант через `transformers`.


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

c:\Users\corey\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\corey\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 10. Генерация ответа модели


In [13]:
def generate_model_response(prompt: str) -> str:
    """Отправляет prompt в модель и возвращает сырой текст ответа."""
    messages = [
        {
            "role": "system",
            "content": "Ты аккуратный классификатор звонков. Отвечай только валидным JSON.",
        },
        {"role": "user", "content": prompt},
    ]

    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(rendered_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=False,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return generated_text.strip()

## 11. Безопасный парсинг JSON

Иногда модель может обернуть JSON в ```json ... ```.
Это нужно корректно убрать.


In [14]:
def safe_parse_json(text: str) -> Dict[str, Any]:
    """Пытается безопасно распарсить JSON-ответ модели."""
    cleaned_text = text.strip()
    cleaned_text = re.sub(r"^```json\s*", "", cleaned_text)
    cleaned_text = re.sub(r"^```\s*", "", cleaned_text)
    cleaned_text = re.sub(r"\s*```$", "", cleaned_text)

    return json.loads(cleaned_text)

## 12. Основная функция role inference


In [15]:
def infer_roles_with_llm(call: CallRecord) -> RoleInferenceResult:
    """Определяет роли в звонке с помощью LLM."""
    prompt = build_role_inference_prompt(call)
    raw_output = generate_model_response(prompt)

    try:
        parsed = safe_parse_json(raw_output)
    except Exception:
        parsed = {
            "manager_speaker": None,
            "client_speakers": [],
            "gatekeeper_present": None,
            "decision_maker_present": None,
            "call_topology": "unclear",
            "confidence": 0.0,
            "evidence": [],
        }

    result = RoleInferenceResult(
        call_id=call.call_id,
        manager_speaker=parsed.get("manager_speaker"),
        client_speakers=parsed.get("client_speakers", []),
        gatekeeper_present=parsed.get("gatekeeper_present"),
        decision_maker_present=parsed.get("decision_maker_present"),
        call_topology=parsed.get("call_topology", "unclear"),
        confidence=float(parsed.get("confidence", 0.0) or 0.0),
        evidence=parsed.get("evidence", []),
        raw_model_output=raw_output,
    )

    return result

## 13. Post-processing после LLM

Этот слой добавляет:
- обработку автоответчика;
- fallback-эвристику для менеджера;
- согласование списка клиентов.


In [16]:
def postprocess_role_result(
    call: CallRecord, result: RoleInferenceResult
) -> RoleInferenceResult:
    """Применяет простые корректировки после LLM."""

    # Если это автоответчик — фиксируем topology
    if is_autoresponder(call):
        result.call_topology = "voicemail"
        result.confidence = max(result.confidence, 0.95)
        return result

    # Если модель не определила менеджера, пробуем простую эвристику
    if result.manager_speaker is None:
        scores = calculate_manager_hint_scores(call)
        if scores:
            best_speaker, best_score = sorted(
                scores.items(), key=lambda item: item[1], reverse=True
            )[0]
            if best_score >= 2:
                result.manager_speaker = best_speaker
                result.confidence = max(result.confidence, 0.55)

    # Если менеджер найден, остальные спикеры считаем клиентской стороной
    if result.manager_speaker is not None:
        all_speakers = get_unique_speakers(call)
        result.client_speakers = [
            speaker for speaker in all_speakers if speaker != result.manager_speaker
        ]

    return result

## 14. Прогон по одному звонку

Полезно сначала протестировать pipeline на одном примере.


In [17]:
sample_call = calls[0]

if is_broken_call(sample_call):
    print("Call looks broken.")
else:
    result = infer_roles_with_llm(sample_call)
    result = postprocess_role_result(sample_call, result)

    print("=== ROLE INFERENCE RESULT ===")
    print(json.dumps(asdict(result), ensure_ascii=False, indent=2))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== ROLE INFERENCE RESULT ===
{
  "call_id": "043e79e1e4cd518e69aa8613e5e40ab50376289b",
  "manager_speaker": null,
  "client_speakers": [
    "SPEAKER_00"
  ],
  "gatekeeper_present": null,
  "decision_maker_present": null,
  "call_topology": "unclear",
  "confidence": 0.0,
  "evidence": [
    {
      "speaker": "SPEAKER_00",
      "turn_id": 0,
      "reason": "инициирует разговор"
    }
  ],
  "raw_model_output": "{\n  \"manager_speaker\": null,\n  \"client_speakers\": [\"SPEAKER_00\"],\n  \"gatekeeper_present\": null,\n  \"decision_maker_present\": null,\n  \"call_topology\": \"unclear\",\n  \"confidence\": 0.0,\n  \"evidence\": [\n    {\n      \"speaker\": \"SPEAKER_00\",\n      \"turn_id\": 0,\n      \"reason\": \"инициирует разговор\"\n    }\n  ]\n}"
}


## 15. Просмотр первых реплик звонка


In [18]:
def preview_call(call: CallRecord, max_blocks: int = 12) -> str:
    """Печатает первые реплики звонка в удобном виде."""
    lines: List[str] = []

    for block in call.speaker_blocks[:max_blocks]:
        lines.append(f"{block.speaker}: {block.text}")

    return "\n".join(lines)


print(preview_call(calls[0], max_blocks=12))

SPEAKER_00: Алло. Алло. Это было...


## 16. Batch inference по нескольким звонкам


In [19]:
results: List[RoleInferenceResult] = []

for call in tqdm(calls):
    if is_broken_call(call):
        results.append(
            RoleInferenceResult(
                call_id=call.call_id,
                manager_speaker=None,
                client_speakers=[],
                gatekeeper_present=None,
                decision_maker_present=None,
                call_topology="broken_call",
                confidence=0.0,
                evidence=[],
                raw_model_output="",
            )
        )
        continue

    result = infer_roles_with_llm(call)
    result = postprocess_role_result(call, result)
    results.append(result)

results_df = pd.DataFrame([asdict(result) for result in results])
results_df.head()

  0%|          | 0/103 [00:00<?, ?it/s]

,call_id,manager_speaker,client_speakers,gatekeeper_present,decision_maker_present,call_topology,confidence,evidence,raw_model_output
0,043e79e1e4cd518e69aa8613e5e40ab50376289b,None,[SPEAKER_00],None,None,unclear,0.00,"[{'speaker': 'SPEAKER_00', 'turn_id': 0, 'reas...","{\n ""manager_speaker"": null,\n ""client_speak..."
1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,SPEAKER_00,[SPEAKER_01],None,None,manager_to_client,0.80,"[{'speaker': 'SPEAKER_00', 'turn_id': 7, 'reas...","{\n ""manager_speaker"": ""SPEAKER_00"",\n ""clie..."
2,077275e8d5b6b39ffe4f04cf96133a82d7ed5900,SPEAKER_00,[SPEAKER_01],None,None,unclear,0.55,"[{'speaker': 'SPEAKER_00', 'turn_id': 0, 'reas...","{\n ""manager_speaker"": null,\n ""client_speak..."
3,0b3ee9a49814179ca0f8789fc9823a70e76188ce,None,[SPEAKER_01],True,None,manager_to_gatekeeper,0.80,"[{'speaker': 'SPEAKER_01', 'turn_id': 0, 'reas...","{\n ""manager_speaker"": null,\n ""client_speak..."
4,0b4d2281a5208d9f9cd995a257829231db9bce27,SPEAKER_00,[SPEAKER_01],None,None,unclear,0.40,"[{'speaker': 'SPEAKER_00', 'turn_id': 0, 'reas...","{\n ""manager_speaker"": ""SPEAKER_00"",\n ""clie..."


## 17. Сохранение результатов


In [20]:
OUTPUT_PATH = Path(
    "data/role_inference_results" + MODEL_NAME.replace("/", "-") + ".jsonl"
)

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(asdict(result), ensure_ascii=False) + "\n")

print(f"Saved results to: {OUTPUT_PATH}")

Saved results to: data\role_inference_resultsQwen-Qwen2.5-3B-Instruct.jsonl


## 18. Что делать дальше

После этого ноутбука следующий логичный шаг:
1. добавить `call_type_classifier`;
2. вынести `role_inference` в отдельный модуль;
3. начать строить event extraction;
4. затем перейти к оценке блоков по вашим критериям.
